<a href="https://colab.research.google.com/github/valerio-unifei/ECAA08-2026.2-Projeto/blob/main/etapa-01-logica/03%20-%20Tautologias%20e%20contradi%C3%A7%C3%B5es.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook: Variação lógica das tags e teste das expressões de tautologias e contradições

Este notebook gera valores lógicos aleatórios para todas as variáveis de processo mapeadas no projeto e avalia as expressões lógicas das regras de segurança, permissivos e validações formais.

A ideia é verificar, em um conjunto grande de combinações, se cada expressão é:
- Tautologia: sempre verdadeira
- Contradição: sempre falsa
- Contingente: depende dos valores das entradas


In [3]:
import itertools
import random
import pandas as pd

variaveis = [
    'p1', 't1', 'g1', 'l_low', 'l_high', 'e1', 'v1', 'v2', 'm1', 'a1',
    'f1', 'c1', 'm2', 'v3'
]

# Gera uma combinação aleatória dos valores lógicos das variáveis
def gerar_estado_aleatorio():
    return {v: random.choice([True, False]) for v in variaveis}

# Gera todas as combinações possíveis (exaustivo)
todas_as_variacoes = [
    dict(zip(variaveis, valores))
    for valores in itertools.product([False, True], repeat=len(variaveis))
]

print(f'Número total de combinações: {len(todas_as_variacoes)}')
print('Exemplo de estado aleatório:')
print(gerar_estado_aleatorio())


Número total de combinações: 16384
Exemplo de estado aleatório:
{'p1': True, 't1': False, 'g1': True, 'l_low': True, 'l_high': False, 'e1': False, 'v1': False, 'v2': False, 'm1': False, 'a1': True, 'f1': True, 'c1': True, 'm2': True, 'v3': False}


## Expressões lógicas do processo

As regras abaixo seguem o conteúdo do material de lógica proposicional do projeto.

In [4]:
def F(vars_):
    return vars_['p1'] or vars_['t1'] or vars_['g1'] or vars_['e1']

def intertravamento_reator(vars_):
    return (not F(vars_)) or (not vars_['v1'] and not vars_['v2'] and vars_['a1'])

def P_open(vars_):
    return (not vars_['p1']) and (not vars_['t1']) and (not vars_['g1']) and (not vars_['l_high']) and vars_['m1']

def permissivo_partida(vars_):
    return (not vars_['v1']) or P_open(vars_)

def agitador(vars_):
    return (not (vars_['l_low'] or vars_['e1'])) or (not vars_['m1'])

def granulador(vars_):
    return (not vars_['v3']) or (vars_['m2'] and vars_['c1'] and vars_['f1'])

def risco_estado(vars_):
    return vars_['p1'] and vars_['v1']

def regra_segurança(vars_):
    return (not vars_['p1']) or (not vars_['v1'])

def prova_contradicao(vars_):
    return risco_estado(vars_) and regra_segurança(vars_)

# Exemplos clássicos de tautologia e contradição
def tautologia_clasica(vars_):
    return vars_['p1'] or (not vars_['p1'])

def contradicao_clasica(vars_):
    return vars_['p1'] and (not vars_['p1'])

# Dicionário com as expressões a serem avaliadas
expressoes = {
    'A. Intertrava de trip do reator': intertravamento_reator,
    'B. Permissivo de partida do reator': permissivo_partida,
    'C. Desligamento do agitador': agitador,
    'D. Bloqueio da granulação': granulador,
    'Prova de segurança (p1 and v1) AND (p1 -> not v1)': prova_contradicao,
    'Tautologia clássica': tautologia_clasica,
    'Contradição clássica': contradicao_clasica
}

def classificar_expressao(fn):
    valores = [fn(v) for v in todas_as_variacoes]
    if all(valores):
        return 'Tautologia'
    elif not any(valores):
        return 'Contradição'
    return 'Contingente'

resultado = []
for nome, fn in expressoes.items():
    valores = [fn(v) for v in todas_as_variacoes]
    resultado.append({
        'Expressão': nome,
        'Verdadeiras': sum(valores),
        'Falsas': len(valores) - sum(valores),
        'Classificação': classificar_expressao(fn)
    })

print(pd.DataFrame(resultado).to_string(index=False))


                                        Expressão  Verdadeiras  Falsas Classificação
                  A. Intertrava de trip do reator         2944   13440   Contingente
               B. Permissivo de partida do reator         8448    7936   Contingente
                      C. Desligamento do agitador        10240    6144   Contingente
                        D. Bloqueio da granulação         9216    7168   Contingente
Prova de segurança (p1 and v1) AND (p1 -> not v1)            0   16384   Contradição
                              Tautologia clássica        16384       0    Tautologia
                             Contradição clássica            0   16384   Contradição


## Simulação aleatória de estados do processo

Abaixo são gerados alguns estados aleatórios para visualização do comportamento das regras.

In [7]:
for i in range(5):
    estado = gerar_estado_aleatorio()
    print(f'Estado #{i+1}: {estado}')
    print('F =', F(estado))
    print('Intertravamento do reator =', intertravamento_reator(estado))
    print('Permissivo de partida =', permissivo_partida(estado))
    print('Desligamento do agitador =', agitador(estado))
    print('Bloqueio da granulação =', granulador(estado))
    print('Prova de segurança =', prova_contradicao(estado))


Estado #1: {'p1': True, 't1': True, 'g1': False, 'l_low': True, 'l_high': False, 'e1': False, 'v1': True, 'v2': False, 'm1': False, 'a1': False, 'f1': True, 'c1': False, 'm2': False, 'v3': True}
F = True
Intertravamento do reator = False
Permissivo de partida = False
Desligamento do agitador = True
Bloqueio da granulação = False
Prova de segurança = False
Estado #2: {'p1': True, 't1': True, 'g1': True, 'l_low': False, 'l_high': False, 'e1': True, 'v1': True, 'v2': True, 'm1': False, 'a1': True, 'f1': False, 'c1': True, 'm2': True, 'v3': False}
F = True
Intertravamento do reator = False
Permissivo de partida = False
Desligamento do agitador = True
Bloqueio da granulação = True
Prova de segurança = False
Estado #3: {'p1': False, 't1': True, 'g1': True, 'l_low': False, 'l_high': True, 'e1': True, 'v1': False, 'v2': True, 'm1': False, 'a1': True, 'f1': True, 'c1': True, 'm2': False, 'v3': False}
F = True
Intertravamento do reator = False
Permissivo de partida = True
Desligamento do agitado

## Interpretação

1. O valor de `Prova de segurança` deve ser sempre `False`, pois representa a condição de risco `p1 and v1` sob a regra de intertravamento `p1 -> not v1`.
2. As regras de processo (intertravamentos e permissivos) são, em geral, formulas contingentes: dependem da combinação de estados físicos reais.
3. A tautologia clássica `p1 or not p1` é sempre verdadeira; a contradição clássica `p1 and not p1` é sempre falsa.